**Table of contents**<a id='toc0_'></a>    
- [Testing Logger class](#toc1_)    
- [Testing config](#toc2_)    
- [Testing main](#toc3_)    
- [Schema Tests](#toc4_)    
- [Database Tests](#toc5_)    

<!-- vscode-jupyter-toc-config
	numbering=false
	anchor=true
	flat=false
	minLevel=1
	maxLevel=6
	/vscode-jupyter-toc-config -->
<!-- THIS CELL WILL BE REPLACED ON TOC UPDATE. DO NOT WRITE YOUR TEXT IN THIS CELL -->

In [ ]:
import sys
import os

sys.path.insert(0, os.path.abspath("../backend"))

try:
    from backend.log.logger import Logger
    import backend.app.config as config
    import backend.app.main as main
    from backend.app.schemas import tenant
except:
    raise ModuleNotFoundError

# <a id='toc1_'></a>[Testing Logger class](#toc0_)
- log(message: str, level: str = "INFO") from Logger class takes 2 arguments
    * message is the string literal to be logged
    * level is a positional argument which takes 3 levels INFO, DEBUG, ERROR  <NOTE: attempts at any other level will exit the processes with status code 1
- read_log(level: str = "INFO") takes level as a positional argument and prints only logs of that level (Defaut level = INFO) 

In [2]:
# logger

Logger = Logger()
Logger.log("Test logging", "INFO")

Logger.read_log(level="ERROR")


Under touch backend/log/log.txt
[2026-09-19 19:12:46], ERROR: .



# <a id='toc2_'></a>[Testing config](#toc0_)
* NOTE config will grow as the project grows and config only contains resuable constants

In [3]:
print(config.WORKSPACE)

/home/muhammad-anas-khan/Desktop/ai_agent_factory


# <a id='toc3_'></a>[Testing main](#toc0_)
main.py is the app entry point in this test only health will be checked as router logic has not been implemented yet.

In [8]:
from fastapi.testclient import TestClient

try:
    app_instance = main.app
    with TestClient(app_instance) as client:
        response = client.get("/")

    print("\n--- TEST SUCCESSFUL ---")
    print(f"Status Code: {response.status_code}")
    print(f"JSON Response: {response.json()}")

except Exception as e:
    print("\n--- DETECTED APP CRASH ON STARTUP ---")
    raise e



--- TEST SUCCESSFUL ---
Status Code: 200
JSON Response: {'message': 'Healthy'}


# <a id='toc4_'></a>[Schema Tests](#toc0_)
* Tenant
* Chat
* Agent

In [1]:
from types import SimpleNamespace

import pytest
from pydantic import ValidationError

from backend.app.core.enums import SourceType
from backend.app.schemas.agent import AgentResponse, CreateAgent
from backend.app.schemas.chat import ChatRequest, ChatResponse, Citation
from backend.app.schemas.tenant import TenantRequest

In [2]:
def test_tenant_valid():
    t = TenantRequest(organization_name="  Acme  ", admin_email="a@example.com")
    assert t.organization_name == "Acme"


def test_tenant_whitespace_name_rejected():
    with pytest.raises(ValidationError):
        TenantRequest(organization_name="   ", admin_email="a@example.com")


def test_tenant_bad_email_rejected():
    with pytest.raises(ValidationError):
        TenantRequest(organization_name="Acme", admin_email="not-an-email")


@pytest.mark.parametrize("url", ["example.com", "ftp://example.com"])
def test_agent_bad_url_rejected(url):
    with pytest.raises(ValidationError):
        CreateAgent(agent_name="Bot", website_url=url)


def test_agent_response_processing_no_prompt():
    r = AgentResponse(agent_id="1", agent_name="Bot", ingestion_status="processing")
    assert r.system_prompt is None


def test_agent_response_from_object():
    obj = SimpleNamespace(
        agent_id="1", agent_name="Bot", ingestion_status="ready",
        system_prompt="hi", failure_reason=None, indexed_chunk_count=3,
        sources=[], created_at=None,
    )
    assert AgentResponse.model_validate(obj).indexed_chunk_count == 3


def test_agent_response_bad_status_rejected():
    with pytest.raises(ValidationError):
        AgentResponse(agent_id="1", agent_name="Bot", ingestion_status="complet")


def test_chat_request_without_session():
    assert ChatRequest(message="hello", session_id='123')


def test_chat_message_too_long_rejected():
    with pytest.raises(ValidationError):
        ChatRequest(message="x" * 10001)


def test_mixed_citations():
    r = ChatResponse(
        answer="...", session_id="abc",
        sources=[
            Citation(source_type=SourceType.DOCUMENT, title="Handbook", page=3),
            Citation(source_type=SourceType.WEBSITE, title="Home", url="https://example.com"),
        ],
    )
    assert len(r.sources) == 2
if __name__ == "__main__":
    test_cases = [
        test_tenant_valid,
        test_tenant_whitespace_name_rejected,
        test_tenant_bad_email_rejected,
        lambda: test_agent_bad_url_rejected("example.com"),
        lambda: test_agent_bad_url_rejected("ftp://example.com"),
        test_agent_response_processing_no_prompt,
        test_agent_response_from_object,
        test_agent_response_bad_status_rejected,
        test_chat_request_without_session,
        test_chat_message_too_long_rejected,
        test_mixed_citations,
    ]

    for index, test_case in enumerate(test_cases, start=1):
        test_case()
        print(f"Test {index}/{len(test_cases)} Passed ✅")

    print("\n--- ALL SCHEMA TESTS PASSED ---")

Test 1/11 Passed ✅
Test 2/11 Passed ✅
Test 3/11 Passed ✅
Test 4/11 Passed ✅
Test 5/11 Passed ✅
Test 6/11 Passed ✅
Test 7/11 Passed ✅
Test 8/11 Passed ✅
Test 9/11 Passed ✅
Test 10/11 Passed ✅
Test 11/11 Passed ✅

--- ALL SCHEMA TESTS PASSED ---


# <a id='toc5_'></a>[Database Tests](#toc0_)
* Creating an engine
* Inserting a tenant
* Dupliacte email handeling 
* Duplicate key hash

In [ ]:
from backend.app.db.database import get_db, get_session, init_db, engine 
from sqlalchemy import inspect 

init_db() 
inspector = inspect(engine) 

print("=== DATABASE SCHEMA HIERARCHY & RELATIONATIONS ===\n")
table_names = inspector.get_table_names()
print(f"Total Tables Found: {len(table_names)} -> {table_names}\n")

for table_name in table_names:
    print(f"Table: {table_name}")
    print("-" * (7 + len(table_name)))
    
    pk_constraint = inspector.get_pk_constraint(table_name)
    primary_keys = pk_constraint.get('constrained_columns', [])
    foreign_keys = inspector.get_foreign_keys(table_name)
    fk_mapping = {}
    
    for fk in foreign_keys:
        referred_table = fk['referred_table']
        referred_columns = fk['referred_columns']
        constrained_columns = fk['constrained_columns']
        
        for parent_col, ref_col in zip(constrained_columns, referred_columns):
            fk_mapping[parent_col] = f"-> REFERENCES {referred_table}({ref_col})"

    columns = inspector.get_columns(table_name)
    print("  Columns:")
    for col in columns:
        col_name = col['name']
        col_type = str(col['type'])
        nullable = "NULL" if col.get('nullable', True) else "NOT NULL"
        
        modifiers = []
        if col_name in primary_keys:
            modifiers.append("[PK]")
        if col_name in fk_mapping:
            modifiers.append(f"[FK {fk_mapping[col_name]}]")
            
        modifier_str = " ".join(modifiers) if modifiers else ""
        print(f"    • {col_name:<20} {col_type:<15} {nullable:<10} {modifier_str}")
    
    if foreign_keys:
        print("\n  Relationships (Hierarchies):")
        for fk in foreign_keys:
            print(f"    ↳ Child of '{fk['referred_table']}' via {fk['constrained_columns']} -> {fk['referred_columns']}")
            
    print("\n" + "="*50 + "\n")


=== DATABASE SCHEMA HIERARCHY & RELATIONATIONS ===

Total Tables Found: 5 -> ['agents', 'conversations', 'messages', 'sources', 'tenants']

Table: agents
-------------
  Columns:
    • agent_id             VARCHAR         NOT NULL   [PK]
    • agent_name           VARCHAR         NOT NULL   
    • system_prompt        TEXT            NULL       
    • indexed_chunk_count  INTEGER         NOT NULL   
    • ingestion_status     VARCHAR(10)     NOT NULL   
    • created_at           DATETIME        NOT NULL   
    • failure_reason       TEXT            NULL       
    • website_url          VARCHAR         NOT NULL   
    • tenant_id            VARCHAR         NOT NULL   [FK -> REFERENCES tenants(tenant_id)]

  Relationships (Hierarchies):
    ↳ Child of 'tenants' via ['tenant_id'] -> ['tenant_id']


Table: conversations
--------------------
  Columns:
    • conversation_id      VARCHAR         NOT NULL   [PK]
    • session_id           VARCHAR         NOT NULL   
    • created_at        

In [19]:
# just run the test_models.py  with pytest